# 02 — PIO Sales Exploratory Data Analysis

This notebook analyzes `PIO_Sales_Data` only. It creates in-memory summaries and charts to assess forecasting readiness; it does not use vehicle wholesale/fleet data, create models, or export datasets.

> **Data safety:** Clear all outputs before committing. The committed notebook must contain code and markdown only—no stored raw previews, tables, or charts.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display, Markdown

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.pio_sales_eda import (
    PIO_COLUMNS,
    build_eda_tables,
    load_pio_sales,
    missing_value_summary,
    overview_summary,
    prepare_pio_sales_for_eda,
    quality_check_summary,
)

plt.style.use("seaborn-v0_8-whitegrid")

## 1. Load `PIO_Sales_Data`

The helper finds the single workbook under `data/raw/`, reads only the PIO sheet, validates the expected source fields, and creates an in-memory analysis copy with parsed dates and numeric measures.

In [ ]:
workbook_path, pio_raw = load_pio_sales(PROJECT_ROOT / "data" / "raw")
pio = prepare_pio_sales_for_eda(pio_raw)

print(f"Workbook: {workbook_path.name}")
print(f"PIO shape: {pio_raw.shape}")
print("Columns:")
for column in pio_raw.columns:
    print(f"- {column}")

### Compact field dictionary

These are working business definitions. Items marked as likely or requiring confirmation should not be treated as final sponsor definitions.

In [ ]:
field_dictionary = pd.DataFrame([
    {"source_field": field, "working_definition": definition}
    for field, definition in PIO_COLUMNS.items()
])
display(field_dictionary)

## 2. Structure, data types, missingness, and coverage

In [ ]:
display(Markdown("### Source data types"))
display(
    pio_raw.dtypes.astype(str)
    .rename_axis("column")
    .reset_index(name="dtype")
)

display(Markdown("### Missing values"))
display(missing_value_summary(pio_raw))

overview = overview_summary(pio_raw, pio)
display(Markdown("### Coverage and totals"))
display(pd.DataFrame([
    {"metric": metric, "value": value}
    for metric, value in overview.items()
]))

The average unit price shown here is total revenue divided by total positive quantity context; row-level unit price is calculated only when quantity is greater than zero. Totals are exploratory and should be reconciled with Mobis before being treated as official business totals.

## 3. Monthly completeness, quantity, revenue, and unit price

The completeness table helps identify partial or structurally different months before any forecasting split is designed.

In [ ]:
tables = build_eda_tables(pio, top_n=10)
monthly = tables["monthly"]
display(monthly)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

axes[0].plot(monthly["month"], monthly["quantity"], marker="o", color="#0066A1")
axes[0].set_title("Monthly PIO Installed Quantity")
axes[0].set_ylabel("Quantity")

axes[1].plot(monthly["month"], monthly["revenue"], marker="o", color="#D97706")
axes[1].set_title("Monthly PIO Revenue")
axes[1].set_ylabel("Revenue ($)")
axes[1].set_xlabel("Month")

fig.autofmt_xdate()
fig.tight_layout()
plt.show()

## 4. Model-part sparsity

`active_months` counts distinct valid `YYYYMM` values for each model-part combination. `avg_monthly_quantity` is total quantity divided by active months, so it describes volume during active months rather than across the entire calendar window.

In [ ]:
model_part_sparsity = tables["model_part_sparsity"]

display(Markdown("### Distribution of model-part activity"))
display(
    model_part_sparsity[
        ["active_months", "total_quantity", "total_revenue", "avg_monthly_quantity"]
    ].describe(percentiles=[0.25, 0.5, 0.75, 0.9]).T
)

display(Markdown("### Sample of sparsest model-part combinations"))
display(model_part_sparsity.head(25))

## 5. Brand, model, part, and model-part concentration

In [ ]:
concentration = tables["concentration"].copy()
concentration["top_n_quantity_share_percent"] = (
    concentration["top_n_quantity_share"] * 100
)
concentration["top_n_revenue_share_percent"] = (
    concentration["top_n_revenue_share"] * 100
)
display(Markdown("### Top-10 concentration shares"))
display(concentration[[
    "entity", "top_n", "group_count",
    "top_n_quantity_share_percent", "top_n_revenue_share_percent",
]])

display(Markdown("### Quantity and revenue by brand/company code"))
display(tables["by_brand"])

display(Markdown("### Top models by quantity"))
display(tables["top_models_by_quantity"])

display(Markdown("### Top models by revenue"))
display(tables["top_models_by_revenue"])

display(Markdown("### Top parts by quantity"))
display(tables["top_parts_by_quantity"])

display(Markdown("### Top parts by revenue"))
display(tables["top_parts_by_revenue"])

display(Markdown("### Top model-part combinations by quantity"))
display(tables["top_model_parts_by_quantity"])

display(Markdown("### Top model-part combinations by revenue"))
display(tables["top_model_parts_by_revenue"])

In [ ]:
top_models = tables["top_models_by_revenue"].sort_values("revenue")
top_parts = tables["top_parts_by_revenue"].sort_values("revenue").copy()
top_parts["label"] = (
    top_parts["PIS_PNO"].astype("string").fillna("Missing part")
    + " — "
    + top_parts["Part Description"].astype("string").fillna("Missing description")
)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
axes[0].barh(top_models["Model"].astype("string"), top_models["revenue"], color="#0066A1")
axes[0].set_title("Top 10 Models by Revenue")
axes[0].set_xlabel("Revenue ($)")

axes[1].barh(top_parts["label"], top_parts["revenue"], color="#D97706")
axes[1].set_title("Top 10 Parts by Revenue")
axes[1].set_xlabel("Revenue ($)")

fig.tight_layout()
plt.show()

## 6. Basic quality checks

The unusual-unit-price check flags values outside the global 0.5th–99.5th percentile range. This is only a screening heuristic: legitimate prices differ substantially by accessory, so later cleaning should compare prices within part number and over time.

In [ ]:
quality_checks = quality_check_summary(pio_raw, pio)
display(quality_checks)

## 7. Interpretation and forecasting-readiness notes

### What the data appears to represent

- Each row appears to describe PIO installed quantity and associated sales dollars for a date, company/brand code, vehicle model, model year, and accessory part.
- The table includes both operational dimensions (model, part, quantity) and financial measures (revenue), which supports separate demand and sales-dollar analysis.

### Working assumptions for the next project steps

- `PIS_CMP_KND` is likely a company/brand code. Observed values appear to be H and K; H may include both Hyundai and Genesis because Genesis models appear without a separate G code.
- `PIS_SERI` appears to align closely with the vehicle model code in `Vehicle_Wholesale_Data`. It is useful for validating mappings and variant differences, but should not be used as the only matching key.
- `PIS_PNO` is the stable accessory key, while `Model` is the vehicle model name.
- PIO installed quantity and PIO revenue are both forecasting targets: quantity supports demand/inventory planning, while revenue supports sales planning and reporting.
- The `SumOf...` field names suggest the sheet may already be aggregated; the official row-level grain must be confirmed with Mobis.
- `Vehicle_Wholesale_Data` uses HMA/GMA/KUS report groups and separates variants such as HEV/PHEV/EV/Coupe, so later matching should combine brand, model name, month, and model-code evidence.
- Penetration rate can later be calculated as PIO quantity divided by matched vehicle volume, but only when the vehicle-volume denominator and match are valid and clearly defined.

### Suitability for monthly forecasting

- The explicit `YYYYMM` field and parsed invoice dates make monthly aggregation feasible.
- The monthly completeness table should be used to identify partial first/last months or changing coverage before defining train/test periods.
- The model-part sparsity results determine whether detailed combinations have enough active history; sparse combinations may need aggregation or intermittent-demand treatment later.
- Concentration shares show how much of total quantity and revenue is carried by a small number of models, parts, or combinations, which can guide forecasting hierarchy decisions.

### Issues to clean before modeling

- Resolve invalid or inconsistent dates and verify that invoice date, `YYYYMM`, and `Deliminated Date` agree.
- Define duplicate-row handling and confirm whether negative values represent returns, credits, or corrections.
- Investigate missing model/part identifiers and quantity/revenue mismatches.
- Review unit-price changes within part number rather than applying a global outlier rule.
- Confirm whether model year, vehicle model names, and vehicle model/series codes are stored consistently across variants and over time.

### Still requires confirmation from Mobis

1. What is the exact H/G/K or HMA/GMA/KUS brand mapping?
2. What is the exact operational meaning of `Deliminated Date`?
3. Which date field is official for month-end reporting?
4. What do negative quantity and revenue values represent: returns, cancellations, credits, or corrections?
5. Is zero quantity with positive revenue a valid business case?
6. What is the official row-level grain, and are the source rows already aggregated?

**Stopping point:** no forecasting model, vehicle-volume merge, dashboard, CSV, or processed dataset is created here.

> **Before committing:** clear every output and execution count.